employment years, annual

income, number of open credit accounts, credit history, loan grade

as decided by LendingClub, home ownership, purpose, and the state

of residence

In [142]:
import sys
dice_path = "/Users/volk/Documents/bau24-25/thesis/repos/DiCE-X"
sys.path.insert(0, dice_path)

In [205]:
%load_ext autoreload
%autoreload 2

In [131]:
import pandas as pd
import numpy as np
df = pd.read_csv("lending_club_dataset/loan.csv", low_memory=False)
df.describe(include="all").transpose()

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
id,39717.0,NaN,NaN,NaN,683131.91306,210694.132915,54734.0,516221.0,665665.0,837755.0,1077501.0
member_id,39717.0,NaN,NaN,NaN,850463.559408,265678.307421,70699.0,666780.0,850812.0,1047339.0,1314167.0
loan_amnt,39717.0,NaN,NaN,NaN,11219.443815,7456.670694,500.0,5500.0,10000.0,15000.0,35000.0
funded_amnt,39717.0,NaN,NaN,NaN,10947.713196,7187.23867,500.0,5400.0,9600.0,15000.0,35000.0
funded_amnt_inv,39717.0,NaN,NaN,NaN,10397.448868,7128.450439,0.0,5000.0,8975.0,14400.0,35000.0
...,...,...,...,...,...,...,...,...,...,...,...
tax_liens,39678.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0
tot_hi_cred_lim,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
total_bal_ex_mort,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
total_bc_limit,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [132]:
for col in df.columns:
    print(col)

id
member_id
loan_amnt
funded_amnt
funded_amnt_inv
term
int_rate
installment
grade
sub_grade
emp_title
emp_length
home_ownership
annual_inc
verification_status
issue_d
loan_status
pymnt_plan
url
desc
purpose
title
zip_code
addr_state
dti
delinq_2yrs
earliest_cr_line
inq_last_6mths
mths_since_last_delinq
mths_since_last_record
open_acc
pub_rec
revol_bal
revol_util
total_acc
initial_list_status
out_prncp
out_prncp_inv
total_pymnt
total_pymnt_inv
total_rec_prncp
total_rec_int
total_rec_late_fee
recoveries
collection_recovery_fee
last_pymnt_d
last_pymnt_amnt
next_pymnt_d
last_credit_pull_d
collections_12_mths_ex_med
mths_since_last_major_derog
policy_code
application_type
annual_inc_joint
dti_joint
verification_status_joint
acc_now_delinq
tot_coll_amt
tot_cur_bal
open_acc_6m
open_il_6m
open_il_12m
open_il_24m
mths_since_rcnt_il
total_bal_il
il_util
open_rv_12m
open_rv_24m
max_bal_bc
all_util
total_rev_hi_lim
inq_fi
total_cu_tl
inq_last_12m
acc_open_past_24mths
avg_cur_bal
bc_open_to_buy
bc

In [133]:
import math

In [134]:
def parse_year(year_as_str):
    year = int(year_as_str)
    if year <= 99:
        return 1900 + year if year > 50 else 2000 + year
    return 2000 + year

In [135]:
new_df = pd.DataFrame()
new_df['employment_years'] = df['emp_length']
new_df['num_open_credit_acc'] = df['open_acc']
new_df['annual_income'] = df['annual_inc']

In [136]:
# credit history
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
lending_months = df['issue_d'].apply(lambda x: x.split('-')[0]).__deepcopy__().copy()
issue_month = pd.DataFrame(pd.Index(months).get_indexer(lending_months))
issue_year = pd.DataFrame(df['issue_d'].apply(lambda x: parse_year(x.split('-')[1])).__deepcopy__().copy())
issue_yearmonth = pd.DataFrame(issue_year.values * 100 + issue_month.values)
adjusted_last_ym = issue_yearmonth[0].apply(lambda x: (float(x) / 100.0) * 100 + ((float(x) - math.floor(float(x) / 100.0) * 100) - 1) / 12 * 100) / 100
earliest_credit_line_year = pd.DataFrame(df['earliest_cr_line'].apply(lambda x: parse_year(x.split('-')[1])).__deepcopy__().copy())
earliest_credit_line_months = df['earliest_cr_line'].apply(lambda x: x.split('-')[0]).__deepcopy__().copy()
earliest_credit_line_months = pd.DataFrame(pd.Index(months).get_indexer(earliest_credit_line_months))
adjusted_credit_line_ym = pd.DataFrame(earliest_credit_line_year.values * 100 + \
                        earliest_credit_line_months.apply(
                            lambda x: x - 1
                        ) / 12 * 100) / 100
adjusted_credit_line_ym = adjusted_credit_line_ym.apply(lambda x: round(x, 2))
adjusted_last_ym = adjusted_last_ym.apply(lambda x: round(x, 2)).to_frame()
credit_ym = (adjusted_last_ym - adjusted_credit_line_ym).apply(lambda x: round(x, 1))
new_df['credit_history'] = credit_ym.values

In [137]:
new_df['loan_grade'] = df['grade']
new_df['home'] = np.where(
    df['home_ownership'].isin(['ANY', 'NONE']), 
    'OTHER', 
    df['home_ownership']
)

# purpose
conditions = [
    df['purpose'].isin(['credit_card', 'debt_consolidation']),  # Condition for "debt"
    df['purpose'].isin(['car', 'major_purchase', 'vacation', 'wedding', 'medical', 'other']),  # Condition for "purchase"
    df['purpose'].isin(['house', 'home_improvement', 'moving', 'renewable_energy'])  # Another condition for "purchase"
]

outputs = ['debt', 'purchase', 'purchase']

new_df['purpose'] = np.select(conditions, outputs, default=df['purpose'])

# state

new_df['addr_state'] = df['addr_state']

In [138]:
new_df.replace('n/a', np.nan,inplace=True)
new_df['employment_years'].fillna(value=0,inplace=True)
new_df['employment_years'].replace(to_replace='[^0-9]+', value='', inplace=True, regex=True)
new_df['employment_years'] = new_df['employment_years'].astype(int)
new_df.head()

,employment_years,num_open_credit_acc,annual_income,credit_history,loan_grade,home,purpose,addr_state
0,10,3,24000.0,27.0,B,RENT,debt,AZ
1,1,3,30000.0,12.8,C,RENT,purchase,GA
2,10,2,12252.0,10.2,C,RENT,small_business,IL
3,10,10,49200.0,15.9,C,RENT,purchase,CA
4,1,15,80000.0,16.0,B,RENT,purchase,OR


In [140]:
ls_conditions = [
    df['loan_status'].isin(['Charged Off', 'Current']),
    df['loan_status'] == 'Fully Paid'
]
ls_outputs = [0, 1]

new_df['loan_status'] = np.select(ls_conditions, ls_outputs, default=df['loan_status'])

new_df.head()

,employment_years,num_open_credit_acc,annual_income,credit_history,loan_grade,home,purpose,addr_state,loan_status
0,10,3,24000.0,27.0,B,RENT,debt,AZ,1
1,1,3,30000.0,12.8,C,RENT,purchase,GA,0
2,10,2,12252.0,10.2,C,RENT,small_business,IL,1
3,10,10,49200.0,15.9,C,RENT,purchase,CA,1
4,1,15,80000.0,16.0,B,RENT,purchase,OR,0


In [143]:
from dice_ml_x.utils import helpers

h_df = helpers.load_lending_club_dataset()
h_df.head()

,employment_years,num_open_credit_acc,annual_income,loan_grade,credit_history,purpose,home,addr_state,loan_status
0,10,3,24000.0,B,27.0,debt,RENT,AZ,1
1,1,3,30000.0,C,12.8,purchase,RENT,GA,0
2,10,2,12252.0,C,10.2,small_business,RENT,IL,1
3,10,10,49200.0,C,15.9,purchase,RENT,CA,1
4,1,15,80000.0,B,16.0,purchase,RENT,OR,0


In [147]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

# Original dataset
df = pd.DataFrame({
    "A": ["cat", "dog", "bird", "cat", "dog"],
    "B": [6, 7, 8, 9, 10]
})

# Encode column "A" using OneHotEncoder
encoder = OneHotEncoder(sparse_output=False)
encoded_A = encoder.fit_transform(df[["A"]])
encoded_df = pd.DataFrame(encoded_A, columns=encoder.get_feature_names_out(["A"]))

# Combine with original numerical column "B"
encoded_df["B"] = df["B"]

# Split original and encoded datasets
train_original, test_original = train_test_split(df, test_size=0.4, random_state=42)
train_encoded, test_encoded = train_test_split(encoded_df, test_size=0.4, random_state=42)

# Print results
print("Train Original:\n", train_original)
print("Train Encoded:\n", train_encoded)
print("Test Original:\n", test_original)
print("Test Encoded:\n", test_encoded)

Train Original:
       A  B
2  bird  8
0   cat  6
3   cat  9
Train Encoded:
    A_bird  A_cat  A_dog  B
2     1.0    0.0    0.0  8
0     0.0    1.0    0.0  6
3     0.0    1.0    0.0  9
Test Original:
      A   B
1  dog   7
4  dog  10
Test Encoded:
    A_bird  A_cat  A_dog   B
1     0.0    0.0    1.0   7
4     0.0    0.0    1.0  10


In [215]:
from dice_ml_x.utils.neuralnetworks import PYTDataset, PYTModel
from torch.utils.data import DataLoader
lending_df = helpers.load_lending_club_dataset()
pyt_train_ds = PYTDataset(lending_df, 'loan_status', train=True)
pyt_test_ds = PYTDataset(lending_df, 'loan_status', train=False)
pyt_train_dt_loader = DataLoader(pyt_train_ds, 16, shuffle=True)
pyt_test_dt_loader = DataLoader(pyt_test_ds, 4, shuffle=False)
dummy_input, dummy_label = next(iter(pyt_train_dt_loader))
in_features = dummy_input.shape[1]
pyt_model = PYTModel(in_features)
pyt_model.train(train_dataloader=pyt_train_dt_loader, test_dataloader=pyt_test_dt_loader)

In [216]:
pyt_model.history['train_acc']

[0.8249716731713458,
 0.8291892232154098,
 0.829566914264132,
 0.8294410172478912,
 0.8295039657560116,
 0.8296298627722523,
 0.8298816568047337,
 0.8305426161399975,
 0.8307944101724789,
 0.8306999874102984]